# Pre-body

## Clearing past runs (optional)

In [1]:
# !rm -rf logs/ # clear logs
# !rm -rf optimizer_output/

## IIC-OSIC Env Setup

In [2]:
# # The following workaround is needed to run the jupyter notebook in docker container.
import os
import subprocess

# Run the script in a shell, capture its environment
PDK = "ihp-sg13g2"
command = f"bash -c 'source /foss/tools/sak/iic-pdk-script.sh {PDK} && source ~/.bashrc && env'"
result = subprocess.run(command, capture_output=True, text=True, shell=True)

# Parse environment variables from the output
for line in result.stdout.splitlines():
    key, _, value = line.partition("=")
    if key and value:
        os.environ[key] = value
        
os.environ["PATH"] += ":/foss/tools/bin"

# Now they are in your current Python process
print("PDK_ROOT:", os.environ.get("PDK_ROOT"))
print("SPICE_USERINIT_DIR:", os.environ.get("SPICE_USERINIT_DIR"))

# Test ngspice
!ngspice -v

PDK_ROOT: /foss/pdks
SPICE_USERINIT_DIR: /foss/pdks/ihp-sg13g2/libs.tech/ngspice
******
** ngspice-44.2 : Circuit level simulation program
** Compiled with KLU Direct Linear Solver
** The U. C. Berkeley CAD Group
** Copyright 1985-1994, Regents of the University of California.
** Copyright 2001-2024, The ngspice team.
** Please get your ngspice manual from https://ngspice.sourceforge.io/docs.html
** Please file your bug-reports at http://ngspice.sourceforge.net/bugrep.html
** Creation Date: Sat May 24 09:38:33 UTC 2025
******


## Library Imports

In [3]:
import logging

from pathlib import Path

from symxplorer.spice_engine                import Spicelib_Wrapper, Sim_Execution_Type
from symxplorer.designer_tools              import Nevergrad_Spice_Multi_Spec_Constraint_Satisfaction, Project_Setup
from symxplorer.logging                     import setup_loggers

logger = logging.getLogger("SymXplorer.jupyter")
logger.info("Spicelib_Wrapper imported successfully.")

2025-10-01 21:03:20,862 - SymXplorer.optimizer - Using device: cpu and dtype: torch.float64
2025-10-01 21:03:20,867 - SymXplorer.jupyter - Spicelib_Wrapper imported successfully.


# Instantiations


## Loading the project config

In [4]:
# ----------------------------
# Instantiations
# ----------------------------
project_setup_yaml = Path(f"/foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml")
_ = setup_loggers()

# (1) Load the project setup information
PROJECT_SETUP = Project_Setup.from_yaml(project_setup_yaml)
PROJECT_SETUP

21:03:20 - SymXplorer: [INFO] 🚀 Logger initialized and ready!
21:03:20 - SymXplorer: [INFO] 📄 Log file: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/logs/SymXplorer_2025-10-01_21-03-20.log
21:03:20 - SymXplorer: [INFO] 🔧 spicelib logger set to 50
21:03:20 - SymXplorer.domains: [INFO] 📂 Loading project setup from /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/project_setup.yaml
21:03:21 - SymXplorer.domains: [INFO] Initialized OptimizerConfig: CMA, type=nevergrad, budget=10, random_seed=48
21:03:21 - SymXplorer.domains: [INFO] 	Linear bounds: min=0, max=100
21:03:21 - SymXplorer.domains: [INFO] 	Log bounds: min=1, max=100
21:03:21 - SymXplorer.domains: [INFO] 	Loss function: max_loss=inf, norm_method=min-max, type=mse, rescale_mag=True, include_phase_loss=False, include_mag_loss=True
21:03:21 - SymXplorer.domains: [INFO] 	Number of target specs: 3
21:03:21 - SymXplorer.domains: [INFO] 		- TargetSpec(name=fc, target=100e6, tolerance=100000.0, goal=exact, 

Project_Setup(name='Tunable-TIA', description='Tunable TIA BPF example sizing in the ihp-sg13g2 technology', simulator='ngspice', ws_root=PosixPath('/foss/designs/eda/SymXplorer'), netlist=PosixPath('examples/tunable-tia/ihp-sg13g2/spice/tb_ac.spice'), outdir=PosixPath('examples/tunable-tia/scripts/optimizer_output'), tech_spec=TechSpec(name='ihp-sg13g2', constraints={'max_nfet_w': np.float64(9.999999999999999e-06), 'min_nfet_w': np.float64(1.8e-07), 'max_nfet_l': np.float64(9.999999999999999e-06), 'min_nfet_l': np.float64(1.8e-07), 'max_pfet_w': np.float64(9.999999999999999e-06), 'min_pfet_w': np.float64(1.8e-07), 'max_pfet_l': np.float64(9.999999999999999e-06), 'min_pfet_l': np.float64(1.8e-07), 'max_cap_w': np.float64(0.01), 'min_cap_w': np.float64(1e-06), 'max_cap_l': np.float64(0.01), 'min_cap_l': np.float64(1e-06), 'max_res_w': np.float64(0.001), 'min_res_w': np.float64(1e-06), 'max_res_l': np.float64(0.001), 'min_res_l': np.float64(1e-06)}), pvt=PVT(temp=25, corner='tt', supply=

## Create a SPICE simulator wrapper

In [5]:
# (2) Create the Spice Simulator Wrapper
netlist_filename = Path(PROJECT_SETUP.ws_root) / Path(PROJECT_SETUP.netlist)
output_folder    = Path(PROJECT_SETUP.ws_root) / Path(PROJECT_SETUP.outdir)

wrapper = Spicelib_Wrapper(
    project_name=PROJECT_SETUP.name,
    netlist_filename=netlist_filename,
    output_folder=output_folder,
    sim_execution_t=Sim_Execution_Type.RUN_AND_WAIT,  # only RUN_AND_WAIT is supported as of now...,
    path_to_simulator=Path("/foss/tools/bin/ngspice"),
    verbose=False
    )
wrapper

21:03:21 - SymXplorer.spicelib: [WARNING] ⚠️ Output directory already exists, re-creating: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output
21:03:21 - SymXplorer.spicelib: [INFO] --------------------------------------------------
21:03:21 - SymXplorer.spicelib: [INFO] 🚀 Spicelib_Wrapper initialized successfully!
21:03:21 - SymXplorer.spicelib: [INFO] 	📝 Project: Tunable-TIA
21:03:21 - SymXplorer.spicelib: [INFO] 	📜 Schematic: tb_ac
21:03:21 - SymXplorer.spicelib: [INFO] 	📂 Output Folder: /foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output
21:03:21 - SymXplorer.spicelib: [INFO] --------------------------------------------------
21:03:21 - SymXplorer.spicelib: [INFO] Using ngspice from ['/foss/tools/bin/ngspice']
21:03:21 - SymXplorer.spicelib: [INFO] 📊 --- Circuit Information ---
21:03:21 - SymXplorer.spicelib: [INFO] 🔗 Nodes in the netlist: ['VSS', 'GND', 'VDD', 'Vbias', 'Von', 'Vop', 'In', 'Ip']
21:03:21 - SymXplorer.spicelib: [INFO] Te

## Create an optimizer object

In [6]:
circuit_optimizer = Nevergrad_Spice_Multi_Spec_Constraint_Satisfaction(
    spicelib_wrapper=wrapper,
    setup_obj=PROJECT_SETUP
)
circuit_optimizer

21:03:21 - SymXplorer.optimizer: [INFO] Initialized the Nevergrad_Spice_Multi_Spec_Optimizer with 3 target specs


## Sanity Check

In [7]:
# wrapper.run_sanity_check(
#     use_editor=True,
#     sim_execution_t=Sim_Execution_Type.RUN_NOW
# )

# Main Body

## Optimization

In [8]:
circuit_optimizer.parameterize()

Dict(vbias=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_cap_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_cap_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_3_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_3_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_s_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_s_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}]):{'x_dut_nfet_w': 50.0, 'x_dut_nfet_l': 50.0, 'x_dut_cap_w': 50.0, 'x_dut_cap_l': 50.0, 'x_dut_res_s_l': 50.0, 'x_dut_res_s_w': 50.0, 'x_dut_res_3_l': 50.0, 'x_dut_res_3_w': 50.0, 'vbias': 50.0}

In [9]:
circuit_optimizer.optimize()

21:03:21 - SymXplorer.optimizer: [INFO] Optimization process started.
21:03:21 - SymXplorer.optimizer: [INFO] Optimizer is set to CMA with budget = 10
Optimizing: 100%|██████████| 10/10 [00:07<00:00,  1.38trial/s]
21:03:28 - SymXplorer.optimizer: [INFO] Optimization process completed.


[{'params': {'x_dut_nfet_w': 41.45509600423837,
   'x_dut_nfet_l': 44.3331731992696,
   'x_dut_cap_w': 52.31494443576946,
   'x_dut_cap_l': 44.60731408530737,
   'x_dut_res_s_l': 46.07246703495674,
   'x_dut_res_s_w': 50.6330213374221,
   'x_dut_res_3_l': 56.81802002351873,
   'x_dut_res_3_w': 49.524988406395515,
   'vbias': 41.13055581881074},
  'loss': np.float64(25.437902179518606),
  'metadata': {'fc': {'curr_val': np.float64(8515673.0),
    'loss': np.float64(0.91484327)},
   'q': {'curr_val': np.float64(9.685618192737099), 'loss': np.float64(0.0)},
   'gain_db': {'curr_val': np.float64(-22.639360034430247),
    'loss': np.float64(24.523058909518607)}}},
 {'params': {'x_dut_nfet_w': 57.48675316057237,
   'x_dut_nfet_l': 62.11321743060015,
   'x_dut_cap_w': 52.74099914352111,
   'x_dut_cap_l': 45.122054728801444,
   'x_dut_res_s_l': 55.84508959918686,
   'x_dut_res_s_w': 48.95980140727695,
   'x_dut_res_3_l': 65.74687206852094,
   'x_dut_res_3_w': 47.910808737652964,
   'vbias': 57

In [10]:
circuit_optimizer.plot_loss(save_path=project_setup_yaml.parent / "loss_curve.html", show=True)

21:03:29 - SymXplorer.optimizer: [INFO] 📊 Plot saved to /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/loss_curve.html
21:03:29 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


## Inspection & Visualization

### (1) Best Param

In [11]:
out = circuit_optimizer.get_best_params()
if out is not None: 
    best_param, loss, metadata = out
metadata

21:03:30 - SymXplorer.optimizer: [INFO] best loss: 21.288228703684027


{'fc': {'curr_val': np.float64(9607475.0), 'loss': np.float64(0.90392525)},
 'q': {'curr_val': np.float64(10.790672207558826), 'loss': np.float64(0.0)},
 'gain_db': {'curr_val': np.float64(-17.109443637540746),
  'loss': np.float64(20.384303453684026)}}

In [12]:
# Print parameter sizes (convert to u)
for param in best_param:
    print(f"{param}: {best_param[param]*1e6 :0.2f}")

x_dut_nfet_w: 5.09
x_dut_nfet_l: 4.61
x_dut_cap_w: 5591.94
x_dut_cap_l: 3277.07
x_dut_res_s_l: 557.80
x_dut_res_s_w: 545.91
x_dut_res_3_l: 515.37
x_dut_res_3_w: 502.93
vbias: 1120276.11


In [13]:
circuit_optimizer.plot_solution(best_param, show_plot=True, trace_name="vout")

21:03:31 - SymXplorer.optimizer: [INFO] total loss: 21.288228703684027
21:03:31 - SymXplorer.optimizer: [INFO] 	Spec 'fc': curr_val=9607475.0, loss=0.90392525
21:03:31 - SymXplorer.optimizer: [INFO] 	Spec 'q': curr_val=10.790672207558826, loss=0.0
21:03:31 - SymXplorer.optimizer: [INFO] 	Spec 'gain_db': curr_val=-17.109443637540746, loss=20.384303453684026


### (3) Metric Trace

In [14]:
circuit_optimizer.plot_optimization_trace(metric_x='fc', metric_y='gain_db', show=True)

21:03:32 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


(tensor([ 8515673.0000,  8432553.0000,  7641115.0000,  9607475.0000,
         10277298.0000,  8032958.0000,  9660656.0000,  8963680.0000,
          9858203.5000,  9547839.0000]),
 tensor([-22.6394, -19.8707, -20.6437, -17.1094, -18.9679, -22.1121, -18.9093,
         -19.2722, -17.8832, -18.6636]))

### (4) Design Space Exploration

In [15]:
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_nfet_w", param_y="x_dut_nfet_l", show=True)
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_cap_l", param_y="x_dut_cap_w", show=True)
circuit_optimizer.plot_design_space_exploration(param_x="vbias", param_y="x_dut_cap_w", show=True)

21:03:32 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


21:03:32 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


21:03:32 - SymXplorer.optimizer: [INFO] Opening interactive plot in browser...


(tensor([0.7404, 1.0287, 0.6714, 1.1203, 0.7904, 1.0361, 0.9350, 0.7430, 0.8499,
         1.0809]),
 tensor([0.0052, 0.0053, 0.0059, 0.0056, 0.0054, 0.0054, 0.0061, 0.0060, 0.0053,
         0.0059]))

# Testing

In [16]:
circuit_optimizer.optimization_log

[{'metric_value': np.float64(25.437902179518606),
  'fit_summary': {'fc': {'curr_val': np.float64(8515673.0),
    'loss': np.float64(0.91484327)},
   'q': {'curr_val': np.float64(9.685618192737099), 'loss': np.float64(0.0)},
   'gain_db': {'curr_val': np.float64(-22.639360034430247),
    'loss': np.float64(24.523058909518607)}},
  'params': {'x_dut_nfet_w': 4.2508904276162075e-06,
   'x_dut_nfet_l': 4.533517608168274e-06,
   'x_dut_cap_w': 0.005231971294132589,
   'x_dut_cap_l': 0.004461285335389885,
   'x_dut_res_s_l': 0.0004612639456792179,
   'x_dut_res_s_w': 0.0005068238831608469,
   'x_dut_res_3_l': 0.0005686120200349521,
   'x_dut_res_3_w': 0.0004957546341798913,
   'vbias': 0.7403500047385934},
  'log': PosixPath('/foss/designs/eda/SymXplorer/examples/tunable-tia/scripts/optimizer_output/run_1/tb_ac_1.log')},
 {'metric_value': np.float64(23.318813977822124),
  'fit_summary': {'fc': {'curr_val': np.float64(8432553.0),
    'loss': np.float64(0.91567447)},
   'q': {'curr_val': np.f

In [17]:
PROJECT_SETUP.optimizer_config.target_specs.list_target_names()

['fc', 'q', 'gain_db']

In [18]:
target_spec = PROJECT_SETUP.optimizer_config.target_specs.get_target_by_name('gain_db')
target_spec

TargetSpec(name='gain_db', target=40, goal=<OptimizationGoalType.EXCEED: 'exceed'>, sim_type=<SimType.AC: 'ac'>, log_scale=False, enable=True, error_type=<Error_Types.RELATIVE_SQUARED: 'relative-squared'>, weight=10.0, tolerance=1, description='gain in dB at fc')

In [19]:
circuit_optimizer.compute_spec_loss(spec_curr_val=-90, target_spec=target_spec)

np.float64(105.625)

In [20]:
PROJECT_SETUP.dut_params

[Param(name='x_dut_nfet_w', min_val=np.float64(1.8e-07), max_val=np.float64(9.999999999999999e-06), val=None, description=None, log_scale=False),
 Param(name='x_dut_nfet_l', min_val=np.float64(1.8e-07), max_val=np.float64(9.999999999999999e-06), val=None, description=None, log_scale=False),
 Param(name='x_dut_cap_w', min_val=np.float64(1e-06), max_val=np.float64(0.01), val=None, description=None, log_scale=False),
 Param(name='x_dut_cap_l', min_val=np.float64(1e-06), max_val=np.float64(0.01), val=None, description=None, log_scale=False),
 Param(name='x_dut_res_s_l', min_val=np.float64(1e-06), max_val=np.float64(0.001), val=None, description=None, log_scale=False),
 Param(name='x_dut_res_s_w', min_val=np.float64(1e-06), max_val=np.float64(0.001), val=None, description=None, log_scale=False),
 Param(name='x_dut_res_3_l', min_val=np.float64(1e-06), max_val=np.float64(0.001), val=None, description=None, log_scale=False),
 Param(name='x_dut_res_3_w', min_val=np.float64(1e-06), max_val=np.fl